# 🤖 GS25 AI Assistant — Train với MiniMind trên Kaggle

## ⚙️ Trước khi chạy:
1. **Settings (⚙️ góc phải)** → **Accelerator** → chọn **GPU P100** ✅
2. **Settings** → **Internet** → bật **ON** ✅ (cần để clone GitHub & tải model)
3. **Add-ons** → **Secrets** → thêm `HF_TOKEN` (token đọc từ huggingface.co/settings/tokens)

## 📦 Upload dataset GS25:
- Vào **Input** → **Upload** → tải file `gs25_qa.jsonl` lên làm Dataset
- Hoặc để trống, notebook sẽ dùng dataset mẫu nhỏ để test

---
**GPU Kaggle:** P100 16GB — đủ train MiniMind 26M thoải mái 🚀

In [ ]:
# ============================================================
# CELL 1: Kiểm tra môi trường Kaggle
# ============================================================
import subprocess, os, sys

# Kiểm tra GPU
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                           '--format=csv,noheader'], capture_output=True, text=True)
print('🖥️  GPU Info:')
print(gpu_info.stdout.strip())

import torch
print(f'\n🔥 PyTorch: {torch.__version__}')
print(f'✅ CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

# Kaggle input path
KAGGLE_INPUT = '/kaggle/input'
WORK_DIR = '/kaggle/working'
print(f'\n📂 Input datasets: {os.listdir(KAGGLE_INPUT) if os.path.exists(KAGGLE_INPUT) else "(trống)"}')

In [ ]:
# ============================================================
# CELL 2: Đọc HF Token từ Kaggle Secrets
# ============================================================
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print(f'✅ Đọc HF_TOKEN thành công: {HF_TOKEN[:10]}...')
except Exception as e:
    print(f'⚠️  Không đọc được secret: {e}')
    print('   → Nhập thủ công bên dưới:')
    HF_TOKEN = input('Nhập HF_TOKEN (hf_...): ').strip()

# Đăng nhập HF
from huggingface_hub import login
if HF_TOKEN and HF_TOKEN.startswith('hf_'):
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Đăng nhập Hugging Face OK')
else:
    print('⚠️  Token rỗng hoặc không hợp lệ — sẽ thử tải công khai')
    HF_TOKEN = None

In [ ]:
# ============================================================
# CELL 3: Clone MiniMind từ GitHub
# ============================================================
import os

MINIMIND_DIR = '/kaggle/working/minimind'

if not os.path.exists(MINIMIND_DIR):
    print('📥 Cloning MiniMind...')
    ret = os.system(f'git clone --depth 1 https://github.com/jingyaogong/minimind.git {MINIMIND_DIR}')
    if ret == 0:
        print('✅ Clone thành công!')
    else:
        raise RuntimeError('❌ Clone thất bại. Kiểm tra Internet đã bật chưa (Settings → Internet → ON)')
else:
    print('✅ MiniMind đã có sẵn')

os.chdir(MINIMIND_DIR)
print(f'📂 Working dir: {os.getcwd()}')
print(f'📋 Files: {[f for f in os.listdir(".") if f.endswith(".py")]}')

In [ ]:
# ============================================================
# CELL 4: Cài dependencies
# ============================================================
print('📦 Cài dependencies...')
os.system('pip install -q transformers==4.44.0 datasets tiktoken sentencepiece accelerate')

if os.path.exists('requirements.txt'):
    os.system('pip install -q -r requirements.txt')
    print('✅ requirements.txt installed')

# Verify
import importlib
for pkg in ['transformers', 'datasets', 'tiktoken', 'sentencepiece']:
    try:
        m = importlib.import_module(pkg)
        print(f'  ✅ {pkg}')
    except ImportError:
        print(f'  ❌ {pkg} — cài thủ công: pip install {pkg}')

In [ ]:
# ============================================================
# CELL 5: Load dataset GS25 (từ Kaggle Input hoặc tạo mẫu)
# ============================================================
import json, os

os.makedirs('dataset', exist_ok=True)
SFT_DATA_PATH = 'dataset/sft_gs25.jsonl'

# Tìm file gs25_qa.jsonl trong Kaggle Input
found_file = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.jsonl'):
            found_file = os.path.join(root, f)
            break

if found_file:
    print(f'✅ Tìm thấy dataset: {found_file}')
    raw_lines = open(found_file, encoding='utf-8').readlines()
else:
    print('⚠️  Không tìm thấy file .jsonl trong Input')
    print('   → Dùng dataset mẫu nhỏ (10 câu) để test pipeline')
    # Dataset mẫu nhỏ để test
    raw_lines = [
        json.dumps({'instruction': 'Part-time tối đa làm bao nhiêu giờ mỗi tuần?',
                    'input': '', 'output': 'Part-time (STPT) tại GS25 bị giới hạn tối đa 23 giờ/tuần và 91 giờ/tháng.'}, ensure_ascii=False),
        json.dumps({'instruction': 'Câu chào khách chuẩn của GS25?',
                    'input': '', 'output': 'Khi khách vào: "GS25 xin chào!". Khi khách ra: "GS25 cảm ơn và hẹn gặp lại!"'}, ensure_ascii=False),
        json.dumps({'instruction': 'Ca đêm GS25 mấy giờ?',
                    'input': '', 'output': 'Ca đêm tại GS25 từ 22:00 đến 06:00 sáng hôm sau (ca 22-6).'}, ensure_ascii=False),
        json.dumps({'instruction': 'Chu kỳ lương GS25 từ ngày mấy?',
                    'input': '', 'output': 'Chu kỳ lương GS25 từ ngày 26 tháng trước đến ngày 25 tháng hiện tại.'}, ensure_ascii=False),
        json.dumps({'instruction': 'Full-time GS25 làm bao nhiêu giờ một tuần?',
                    'input': '', 'output': 'Full-time (STFT) chuẩn 48 giờ/tuần, tương đương 6 ca 8 tiếng và 1 ngày nghỉ OFF.'}, ensure_ascii=False),
    ]

# Chuyển sang format conversations của MiniMind
converted = []
for line in raw_lines:
    item = json.loads(line.strip())
    # Hỗ trợ cả 2 format: {instruction, output} hoặc {conversations}
    if 'conversations' in item:
        converted.append(item)
    else:
        converted.append({
            'conversations': [
                {'role': 'system', 'content': 'Bạn là AI trợ lý nghiệp vụ GS25 Việt Nam. Hãy trả lời chính xác và hữu ích.'},
                {'role': 'user', 'content': item.get('instruction', '')},
                {'role': 'assistant', 'content': item.get('output', '')}
            ]
        })

# Lưu ra file
with open(SFT_DATA_PATH, 'w', encoding='utf-8') as f:
    for d in converted:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')

print(f'✅ Dataset: {len(converted)} samples → {SFT_DATA_PATH}')
# Preview
sample = converted[0]['conversations']
print(f'\n📋 Sample đầu:')
print(f'  Q: {sample[1]["content"]}')
print(f'  A: {sample[2]["content"][:80]}...')

In [ ]:
# ============================================================
# CELL 6: Tải pre-trained MiniMind weights
# ============================================================
from huggingface_hub import snapshot_download
import os

PRETRAIN_DIR = '/kaggle/working/minimind/out/pretrain_base'
os.makedirs(PRETRAIN_DIR, exist_ok=True)

# Thử repo mới nhất: minimind-3o
REPO_OPTIONS = [
    'jingyaogong/minimind-3o',        # Mới nhất (2025)
    'jingyaogong/minimind-3o-pytorch', # PyTorch weights
]

downloaded = False
for repo_id in REPO_OPTIONS:
    try:
        print(f'⬇️  Thử tải: {repo_id} ...')
        snapshot_download(
            repo_id=repo_id,
            local_dir=PRETRAIN_DIR,
            token=HF_TOKEN,
            ignore_patterns=['*.msgpack', '*.h5', 'flax_*', 'tf_*', '*.ot', 'rust_model*']
        )
        print(f'✅ Tải xong từ {repo_id}!')
        downloaded = True
        break
    except Exception as e:
        print(f'  ⚠️  Lỗi: {str(e)[:100]}')
        print(f'  → Thử repo tiếp theo...')

if not downloaded:
    print('\n🔄 Thử tải từ ModelScope (không cần token)...')
    os.system('pip install -q modelscope')
    from modelscope import snapshot_download as ms_dl
    try:
        ms_dl('jingyaogong/minimind-3o', cache_dir=PRETRAIN_DIR)
        print('✅ Tải từ ModelScope thành công!')
        downloaded = True
    except Exception as e:
        print(f'❌ ModelScope cũng lỗi: {e}')
        print('\n💡 Giải pháp: Train từ đầu (scratch) — mất ~2h nhưng không cần weights')

if downloaded:
    files = os.listdir(PRETRAIN_DIR)
    print(f'\n📂 Files trong pretrain_base: {files[:10]}')

In [ ]:
# ============================================================
# CELL 7: Tìm script SFT và chạy Fine-tuning
# ============================================================
import os, subprocess

os.chdir(MINIMIND_DIR)

# Liệt kê các file train
py_files = sorted([f for f in os.listdir('.') if f.endswith('.py')])
train_files = [f for f in py_files if 'train' in f.lower() or 'sft' in f.lower()]
print('📋 Train scripts tìm thấy:', train_files)
print('📋 Tất cả .py files:', py_files)

# Chọn script SFT theo thứ tự ưu tiên
SFT_SCRIPT = None
for candidate in ['train_sft.py', '2-sft.py', 'sft.py', 'train_full_sft.py', 'finetune.py']:
    if os.path.exists(candidate):
        SFT_SCRIPT = candidate
        break

if SFT_SCRIPT:
    print(f'\n✅ Sẽ dùng script: {SFT_SCRIPT}')
    # Xem help để biết các tham số
    result = subprocess.run(['python', SFT_SCRIPT, '--help'],
                           capture_output=True, text=True, timeout=30)
    print('\n📖 Tham số hỗ trợ:')
    print(result.stdout[:1000] if result.stdout else result.stderr[:500])
else:
    print('⚠️  Không tìm thấy script SFT tự động')
    print('   Hãy chọn thủ công từ danh sách:', train_files)
    SFT_SCRIPT = input('Nhập tên script: ').strip()

In [ ]:
# ============================================================
# CELL 8: 🚀 CHẠY SFT FINE-TUNING
# ============================================================
import os, subprocess, time

OUTPUT_DIR = '/kaggle/working/minimind/out/gs25_sft'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tham số train — tối ưu cho Kaggle P100 (16GB)
TRAIN_ARGS = {
    '--data_path': SFT_DATA_PATH,
    '--out_dir': OUTPUT_DIR,
    '--epochs': '20',        # Nhiều epoch vì dataset nhỏ (147 samples)
    '--batch_size': '8',     # P100 16GB dư sức
    '--learning_rate': '3e-5',
    '--max_seq_len': '512',
}

# Build command
cmd = ['python', SFT_SCRIPT]
for k, v in TRAIN_ARGS.items():
    cmd.extend([k, v])

print(f'🚀 Bắt đầu SFT Fine-tuning GS25...')
print(f'   Script: {SFT_SCRIPT}')
print(f'   Data: {SFT_DATA_PATH} ({len(converted)} samples)')
print(f'   Epochs: 20')
print(f'   Output: {OUTPUT_DIR}')
print(f'   Command: {" ".join(cmd)}')
print('='*60)

start = time.time()

# Thực thi
result = subprocess.run(cmd, text=True, cwd=MINIMIND_DIR)

elapsed = time.time() - start
print(f'\n⏱️  Thời gian: {elapsed/60:.1f} phút')

if result.returncode == 0:
    files = os.listdir(OUTPUT_DIR)
    print(f'\n✅ Fine-tuning hoàn thành!')
    print(f'📂 Model files: {files}')
else:
    print(f'\n❌ Lỗi (return code: {result.returncode})')
    print('💡 Thử chạy lại Cell 7 để kiểm tra help của script')

In [ ]:
# ============================================================
# CELL 9: Test inference
# ============================================================
import os, subprocess

# Tìm script inference/chat
infer_scripts = [f for f in os.listdir('.') if any(k in f.lower() for k in ['infer', 'chat', 'eval', 'generate']) and f.endswith('.py')]
print('🔍 Inference scripts:', infer_scripts)

TEST_QUESTIONS = [
    'Part-time tối đa làm bao nhiêu giờ mỗi tuần?',
    'Ca đêm GS25 có phụ cấp không?',
    'Câu chào khách chuẩn của GS25 là gì?',
    'Chu kỳ lương GS25 từ ngày mấy?',
]

if infer_scripts:
    script = infer_scripts[0]
    print(f'\n🧪 Chạy test với {script}...')
    for q in TEST_QUESTIONS[:2]:  # Test 2 câu trước
        print(f'\n❓ {q}')
        result = subprocess.run(
            ['python', script, '--model_path', OUTPUT_DIR, '--question', q],
            capture_output=True, text=True, timeout=60
        )
        print(f'💬 {result.stdout.strip() or result.stderr.strip()}')
else:
    print('ℹ️  Không có script inference. Thử chạy thủ công:')
    print(f'   python [inference_script].py --model_path {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELL 10: ☁️ Upload model lên Hugging Face
# ============================================================
from huggingface_hub import HfApi
import os

# Lấy username từ HF
api = HfApi(token=HF_TOKEN)
try:
    user_info = api.whoami()
    HF_USERNAME = user_info['name']
    print(f'👤 Đã đăng nhập HF: {HF_USERNAME}')
except:
    HF_USERNAME = input('Nhập Hugging Face username: ').strip()

REPO_ID = f'{HF_USERNAME}/gs25-assistant'
print(f'\n📤 Upload lên: https://huggingface.co/{REPO_ID}')

# Tạo repo (nếu chưa có)
api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True, private=False)

# Upload folder
model_files = os.listdir(OUTPUT_DIR)
if model_files:
    print(f'📦 Files sẽ upload: {model_files}')
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=REPO_ID,
        repo_type='model',
        commit_message='GS25 AI Assistant - MiniMind SFT'
    )
    print(f'\n🎉 Upload thành công!')
    print(f'🔗 Model: https://huggingface.co/{REPO_ID}')
    MODEL_API_URL = f'https://api-inference.huggingface.co/models/{REPO_ID}'
    print(f'📡 API URL: {MODEL_API_URL}')
    print()
    print('='*65)
    print('📝 BƯỚC CUỐI — Thêm vào file .env của dự án GS25:')
    print(f'   VITE_GS25_AI_MODEL_URL={MODEL_API_URL}')
    print(f'   VITE_HF_TOKEN={HF_TOKEN}')
    print('='*65)
else:
    print(f'❌ Thư mục {OUTPUT_DIR} trống — kiểm tra lại Cell 8')

In [ ]:
# ============================================================
# CELL 11 (Tùy chọn): Download model về máy
# ============================================================
# Nếu không muốn dùng HF, có thể tải thẳng model .pth về máy
from IPython.display import FileLink
import shutil, os

# Tạo archive
ARCHIVE = '/kaggle/working/gs25_model.tar.gz'
shutil.make_archive('/kaggle/working/gs25_model', 'gztar', OUTPUT_DIR)

if os.path.exists(ARCHIVE):
    size_mb = os.path.getsize(ARCHIVE) / 1e6
    print(f'✅ Model archive: {ARCHIVE} ({size_mb:.1f} MB)')
    print('📥 Click link để tải về:')
    display(FileLink('/kaggle/working/gs25_model.tar.gz'))
else:
    print('❌ Không tạo được archive')